# Hittite Cuneiform Pipeline
## Adapted from ThreeLanguagePipeline_v2 for Hittite

**Data source:** `7000_hitt_txts_wGloss.csv` (Zenodo 14266302)  
**Language:** Hittite  
**Script:** Cuneiform (adapted from Old Babylonian)

This notebook applies the same pipeline used for Akkadian, Sumerian, and Elamite to Hittite cuneiform data.


## 0. Setup & Dependencies

In [1]:
!pip install -q gensim scikit-learn matplotlib seaborn pandas openpyxl regex torch
!pip install -q git+https://anonymous.4open.science/r/cunei-tools

import pandas as pd
import numpy as np
import re
import json
import unicodedata
import warnings
from collections import Counter, defaultdict
from pathlib import Path

warnings.filterwarnings('ignore')
print("Setup complete.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 77.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Setup complete.


## 1. Configuration

Set the paths to your data files. Adjust `BASE_PATH` for your environment (Colab with Google Drive, local, etc.).


In [14]:
# ============================================================
# CONFIGURATION
# ============================================================
# For Google Colab with Drive:
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = 'data/'

# For local:
#BASE_PATH = './'

HITT_PATH = BASE_PATH + '7000_hitt_txts_wGloss.csv'

print(f"Data path: {HITT_PATH}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data path: data/7000_hitt_txts_wGloss.csv


## 2. POS Harmonization

Map Hittite glosses to the unified tagset used across all four languages.


In [15]:
# ============================================================
# POS HARMONIZATION FOR HITTITE
# ============================================================

# Hittite gloss → unified POS mapping
# Glosses in the Hittite data use morphological abbreviations
# e.g., "FNL(u).NOM.SG.C" → noun features, "3SG.PRS" → verb features

def map_hittite_pos(gloss):
    """Map Hittite gloss string to unified POS tag."""
    if pd.isna(gloss) or gloss == '' or gloss == 'nan':
        return 'X'
    g = str(gloss).strip()

    # Named entities / determinatives
    if g.startswith('DN') or g == 'D/L.PL' and False:  # divine names need context
        pass

    # Verb indicators
    verb_markers = ['1SG', '2SG', '3SG', '1PL', '2PL', '3PL',
                    'PRS', 'PST', 'IMP', 'INF', 'PTCP', 'SUP']
    if any(m in g for m in verb_markers):
        return 'VERB'

    # Noun/adjective case markers
    noun_markers = ['NOM', 'ACC', 'GEN', 'DAT', 'LOC', 'ABL', 'INS', 'ALL', 'ERG',
                    'VOC', 'D/L']
    if any(m in g for m in noun_markers):
        # Check for FNL (final/noun) or adjective markers
        if 'FNL' in g:
            return 'NOUN'
        return 'NOUN'

    # Conjunction
    if g == 'CNJ' or g.startswith('CNJ'):
        return 'CONJ'

    # Demonstrative/adverb
    if g.startswith('DEM') or 'DEMadv' in g:
        return 'ADV'

    # Particles and adverbs
    if g in ('PREV', 'PTC', 'NEG', 'QUOT', 'QUES', 'EMPH'):
        return 'MOD'

    # Numbers
    if g.startswith('NUM') or g.isdigit():
        return 'NUM'

    # Pronouns
    if 'PRON' in g or g.startswith('REL') or g.startswith('REFL'):
        return 'PRON'

    # Postpositions
    if g.startswith('POSP') or g.startswith('PREP'):
        return 'ADP'

    # Determinative markers
    if '(UNM)' in g:
        return 'NOUN'

    return 'X'


ENTITY_TAGS = {'PN', 'DN', 'GN', 'CN', 'RN', 'QN', 'WN', 'MN', 'AN', 'FN', 'TN', 'LN', 'ON', 'SN'}

def map_pos_grammatical(pos_unified):
    return 'PROPN' if pos_unified in ENTITY_TAGS else pos_unified

def map_ner_tag(pos_unified):
    return pos_unified if pos_unified in ENTITY_TAGS else 'O'

print("POS harmonization maps loaded for Hittite.")


POS harmonization maps loaded for Hittite.


## 3. Sign Lists & Unicode Conversion

Load Nuolenna + Akkademia sign lists, then apply Hittite-specific preprocessing:
1. Replace `{ }` with whitespace (determinatives become separate tokens)
2. Remove `[ ]` and `⸢ ⸣` (damage markers, no whitespace replacement)
3. Normalize `ḫ → h` and strip diacritical accents for lookup
4. Convert to Unicode signs


In [16]:
# ============================================================
# LOAD SIGN LISTS
# ============================================================
sign_list = pd.read_json(
    'https://raw.githubusercontent.com/situx/Nuolenna/master/sign_list.json',
    orient='index'
)
sign_list.columns = ['unicode']
sign_list['sign'] = sign_list.index.tolist()
sign_list = sign_list[['sign', 'unicode']].reset_index(drop=True)

try:
    akkademia = pd.read_csv(
        'https://raw.githubusercontent.com/gaigutherz/Akkademia/master/cuneiform_to_unicode_fixed.csv'
    )
    merged = pd.merge(sign_list, akkademia, on=['sign', 'unicode'], how='outer')
except:
    merged = sign_list.copy()

sign_dict = dict(zip(merged['sign'].astype(str), merged['unicode'].astype(str)))

# Add lowercase variants
sign_dict_full = {}
for k, v in sign_dict.items():
    sign_dict_full[k] = v
    sign_dict_full[k.lower()] = v
sign_dict = sign_dict_full

# Load manual corrections if available
try:
    UNMATCHED_PATH_1 = BASE_PATH + 'unmatchednew_AAedit - unmatchednew.csv'
    UNMATCHED_PATH_2 = BASE_PATH + 'unmatchednew - solonew.csv'
    unmatched = pd.read_csv(UNMATCHED_PATH_1)[['unmatched_sign','use']].dropna()
    unmatched2 = pd.read_csv(UNMATCHED_PATH_2)[['value', 'SIGN']].dropna()
    manual_dict = dict(zip(unmatched['unmatched_sign'], unmatched['use']))
    manual_dict2 = dict(zip(unmatched2['value'].str.strip("[]' "), unmatched2['SIGN']))
    sign_dict.update(manual_dict)
    sign_dict.update(manual_dict2)
    print(f"Manual corrections loaded: {len(manual_dict)} + {len(manual_dict2)}")
except FileNotFoundError:
    print("Manual correction files not found — using base sign lists only.")

print(f"Total sign mappings: {len(sign_dict)}")


Manual corrections loaded: 114 + 22
Total sign mappings: 18273


In [17]:
# ============================================================
# HITTITE-SPECIFIC UNICODE CONVERSION
# ============================================================

def normalize_for_lookup(tok):
    """Normalize Hittite-specific characters for sign list lookup."""
    t = tok
    # ḫ → h (standard Hittitological convention)
    t = t.replace('ḫ', 'h').replace('Ḫ', 'H')
    # Strip combining diacritical marks (accents: í→i, é→e, etc.)
    t = ''.join(c for c in unicodedata.normalize('NFD', t)
                if unicodedata.category(c) != 'Mn')
    # Remove tilde variants used in some Hittite editions
    t = t.replace('~', '').replace('˽', '')
    # Remove half-brackets
    t = t.replace('⸢', '').replace('⸣', '')
    return t


def lookup_sign(tok, sign_dict):
    """Try multiple normalization strategies for sign lookup."""
    # Direct match
    for t in [tok, tok.lower(), tok.upper()]:
        if t in sign_dict:
            return sign_dict[t]
    # Normalize ḫ and accents
    n = normalize_for_lookup(tok)
    for t in [n, n.lower(), n.upper()]:
        if t in sign_dict:
            return sign_dict[t]
    # Strip subscript numbers (₀₁₂₃₄₅₆₇₈₉)
    stripped = re.sub(r'[₀₁₂₃₄₅₆₇₈₉]+$', '', n)
    for t in [stripped, stripped.lower(), stripped.upper()]:
        if t in sign_dict:
            return sign_dict[t]
    # Strip trailing ASCII digits
    stripped2 = re.sub(r'\d+$', '', n)
    for t in [stripped2, stripped2.lower(), stripped2.upper()]:
        if t in sign_dict:
            return sign_dict[t]
    return None


def preprocess_hittite_translit(translit):
    """Apply Hittite-specific preprocessing per data documentation.

    1. Replace { } with whitespace (determinatives → separate tokens)
    2. Remove [ ] (damage brackets, no whitespace)
    3. Remove ⸢ ⸣ (half-brackets, no whitespace)
    """
    t = str(translit)
    t = t.replace('{', ' ').replace('}', ' ')
    t = t.replace('[', '').replace(']', '')
    t = t.replace('⸢', '').replace('⸣', '')
    t = re.sub(r'\s+', ' ', t).strip()
    return t


def hittite_to_unicode(translit, sign_dict):
    """Convert Hittite transliteration to Unicode cuneiform signs.

    Returns:
        (list_of_unicode_signs, list_of_unmatched_tokens)
    """
    preprocessed = preprocess_hittite_translit(translit)
    # Split on hyphens and dots (sign separators)
    normalized = preprocessed.replace('-', ' ').replace('.', ' ')
    # Remove damage/uncertainty markers
    for ch in ['#', '!', '?', '*', '(', ')', '°', '½']:
        normalized = normalized.replace(ch, '')
    normalized = re.sub(r'\s+', ' ', normalized).strip()

    tokens = normalized.split()
    unicode_signs = []
    unmatched = []

    for tok in tokens:
        tok = tok.strip()
        if not tok:
            continue
        uni = lookup_sign(tok, sign_dict)
        if uni and str(uni) != 'nan':
            unicode_signs.append(uni)
        else:
            unmatched.append(tok)

    return unicode_signs, unmatched


# Test conversion
test_cases = [
    'LUGAL-uš', 'ku-wa-pí', 'DINGIR{MEŠ}-aš',
    '{D}10-aš-pát', '{LÚ}GUDU₁₂', 'a-ru-wa-a-ez-zi',
    'NINDA.GUR₄.RA', 'MUNUS.LUGAL', 'ḫa-an-da-an-za'
]
print("=== Hittite Unicode Conversion Tests ===")
for tc in test_cases:
    signs, unm = hittite_to_unicode(tc, sign_dict)
    status = "✓" if not unm else f"✗ unmatched: {unm}"
    print(f"  {tc:30s} → {len(signs)} signs  {status}")


=== Hittite Unicode Conversion Tests ===
  LUGAL-uš                       → 2 signs  ✓
  ku-wa-pí                       → 3 signs  ✓
  DINGIR{MEŠ}-aš                 → 3 signs  ✓
  {D}10-aš-pát                   → 4 signs  ✓
  {LÚ}GUDU₁₂                     → 2 signs  ✓
  a-ru-wa-a-ez-zi                → 6 signs  ✓
  NINDA.GUR₄.RA                  → 3 signs  ✓
  MUNUS.LUGAL                    → 2 signs  ✓
  ḫa-an-da-an-za                 → 5 signs  ✓


## 4. Load Hittite Data

In [18]:
# ============================================================
# LOAD HITTITE DATA
# ============================================================

raw = pd.read_csv(HITT_PATH)
print(f"Raw data: {len(raw)} rows, {raw['txtid'].nunique()} texts")
print(f"Columns: {list(raw.columns)}")

# Filter out empty/damaged entries
hitt = raw[raw['translit'].notna() & (raw['translit'] != '…')].copy()
print(f"After filtering: {len(hitt)} rows")

# Apply Unicode conversion
print("\nConverting to Unicode...")
conversion_results = hitt['translit'].apply(lambda x: hittite_to_unicode(x, sign_dict))
hitt['unicode_signs'] = conversion_results.apply(lambda x: x[0])
hitt['unmatched'] = conversion_results.apply(lambda x: x[1])
hitt['form_unicode'] = hitt['unicode_signs'].apply(lambda x: ' '.join(x) if x else '')
hitt['n_signs'] = hitt['unicode_signs'].apply(len)
hitt['n_unmatched'] = hitt['unmatched'].apply(len)
hitt['clean'] = hitt['n_unmatched'] == 0

# Conversion stats
total_signs = hitt['n_signs'].sum() + hitt['n_unmatched'].sum()
converted_signs = hitt['n_signs'].sum()
clean_words = hitt['clean'].sum()
print(f"\n=== Conversion Statistics ===")
print(f"Words:  {clean_words}/{len(hitt)} clean ({clean_words/len(hitt)*100:.1f}%)")
print(f"Signs:  {converted_signs}/{total_signs} converted ({converted_signs/total_signs*100:.1f}%)")

# Unmatched analysis
all_unmatched = Counter()
for u in hitt['unmatched']:
    all_unmatched.update(u)
print(f"\nUnique unmatched signs: {len(all_unmatched)}")
print("Top 15 unmatched:")
for s, c in all_unmatched.most_common(15):
    print(f"  {s:20s} {c}")

# Map POS
hitt['pos_unified'] = hitt['gloss'].apply(map_hittite_pos)
hitt['pos_grammatical'] = hitt['pos_unified'].apply(map_pos_grammatical)
hitt['ner_tag'] = hitt['pos_unified'].apply(map_ner_tag)
hitt['form_latin'] = hitt['translit']
hitt['text_id'] = hitt['txtid']
hitt['language'] = 'hit'
hitt['lemma'] = hitt['word']  # use 'word' column as lemma proxy

print(f"\n=== POS Distribution ===")
for pos, cnt in hitt['pos_unified'].value_counts().items():
    print(f"  {pos:10s} {cnt:7,} ({cnt/len(hitt)*100:5.1f}%)")


Raw data: 170496 rows, 7099 texts
Columns: ['Unnamed: 0', 'txtid', 'lnr', 'cth_number', 'word', 'translit', 'gloss', 'trans_de']
After filtering: 170496 rows

Converting to Unicode...

=== Conversion Statistics ===
Words:  168101/170496 clean (98.6%)
Signs:  524495/526981 converted (99.5%)

Unique unmatched signs: 365
Top 15 unmatched:
  waₐ                  384
  ˽                    329
  weₑ                  172
  wiᵢ                  171
  ~IŠTAR~              164
  NA~˽~PA              133
  n+1                  91
  ~A+NA~               50
  DINGIR˽LÚ            42
  wuᵤ                  35
  〈an〉                 31
  〈aš〉                 30
  TU₇˽BA               28
  n+2                  27
  kat+ta               26

=== POS Distribution ===
  NOUN        76,687 ( 45.0%)
  VERB        43,571 ( 25.6%)
  X           41,590 ( 24.4%)
  MOD          4,064 (  2.4%)
  CONJ         2,487 (  1.5%)
  ADP          1,578 (  0.9%)
  ADV            518 (  0.3%)
  PRON             1 (  0.0%)


## 5. Dataset Summary

In [19]:
# ============================================================
# DATASET SUMMARY
# ============================================================

print(f"{'='*60}")
print(f"Dataset: Hittite")
print(f"{'='*60}")
print(f"Tokens:     {len(hitt):,}")
print(f"Texts:      {hitt['text_id'].nunique():,}")
print(f"Vocab:      {hitt['form_latin'].nunique():,}")
print(f"Lemmas:     {hitt['lemma'].nunique():,}")
print(f"CTH nums:   {hitt['cth_number'].nunique():,}")
print(f"")
print(f"Unicode conversion rate: {clean_words/len(hitt)*100:.1f}%")
print(f"Unique Unicode signs: {len(set(s for signs in hitt['unicode_signs'] for s in signs))}")
print(f"Avg signs/word: {hitt['n_signs'].mean():.2f}")


Dataset: Hittite
Tokens:     170,496
Texts:      7,099
Vocab:      50,269
Lemmas:     12,464
CTH nums:   143

Unicode conversion rate: 98.6%
Unique Unicode signs: 360
Avg signs/word: 3.08


## 6. Build Document Corpora

In [20]:
# ============================================================
# BUILD DOCUMENT CORPORA
# ============================================================

# Latin documents (transliteration-based)
docs_latin = {}
for tid, g in hitt.groupby('text_id'):
    docs_latin[tid] = ' '.join(g['form_latin'].dropna().astype(str))

# Unicode documents (only from clean words)
docs_unicode = {}
for tid, g in hitt.groupby('text_id'):
    unicode_words = g[g['clean']]['form_unicode'].dropna()
    if len(unicode_words) > 0:
        docs_unicode[tid] = ' '.join(unicode_words.astype(str))

# Unicode documents for segmentation (word = list of signs)
docs_segmented = {}
for tid, g in hitt.groupby('text_id'):
    words = g[g['clean']]['unicode_signs'].tolist()
    words = [w for w in words if len(w) > 0]
    if len(words) >= 2:
        docs_segmented[tid] = words

print(f"Latin documents:     {len(docs_latin)}")
print(f"Unicode documents:   {len(docs_unicode)}")
print(f"Segmented documents: {len(docs_segmented)}")

doc_corpora = {'hit': {'latin': docs_latin, 'unicode': docs_unicode}}
datasets = {'hit': hitt}


Latin documents:     7099
Unicode documents:   7086
Segmented documents: 5896


## 7. Word Boundary Inference (Transitional Probability)

Transitional probability segmentation on Unicode sign streams.
Same method as Akkadian/Sumerian/Elamite experiments.


In [21]:
# ============================================================
# WORD BOUNDARY INFERENCE — HITTITE
# ============================================================
import random

docs = docs_segmented
doc_ids = sorted(docs.keys())
random.seed(42)
random.shuffle(doc_ids)
fold_size = len(doc_ids) // 5

print(f"Documents for segmentation: {len(doc_ids)}")
total_signs = sum(sum(len(w) for w in docs[d]) for d in doc_ids)
total_words = sum(len(docs[d]) for d in doc_ids)
print(f"Total signs: {total_signs:,}")
print(f"Total words: {total_words:,}")
print(f"Avg signs/word: {total_signs/total_words:.2f}")

def compute_tp(train_ids, docs):
    bi, uni = Counter(), Counter()
    for did in train_ids:
        s = [sign for w in docs[did] for sign in w]
        for i in range(len(s)):
            uni[s[i]] += 1
            if i < len(s) - 1:
                bi[(s[i], s[i+1])] += 1
    return {k: c / uni[k[0]] for k, c in bi.items()}

def evaluate_tp(test_ids, docs, tp, theta):
    tc = fc = fnc = 0
    for did in test_ids:
        words = docs[did]
        s = [sign for w in words for sign in w]
        gold = set()
        pos = 0
        for w in words:
            pos += len(w)
            gold.add(pos)
        gold.discard(pos)  # remove end-of-doc

        pred = {i+1 for i in range(len(s)-1)
                if tp.get((s[i], s[i+1]), 0) < theta}
        tc += len(pred & gold)
        fc += len(pred - gold)
        fnc += len(gold - pred)

    p = tc / (tc + fc) if (tc + fc) else 0
    r = tc / (tc + fnc) if (tc + fnc) else 0
    f1 = 2 * p * r / (p + r) if (p + r) else 0
    return f1, p, r

# 5-fold CV with threshold sweep
print(f"\n{'='*60}")
print(f"  WORD BOUNDARY INFERENCE — HITTITE (5-fold CV)")
print(f"{'='*60}")

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
sweep_results = []

for theta in thresholds:
    fold_f1s, fold_ps, fold_rs = [], [], []
    for fold in range(5):
        ts = fold * fold_size
        te = ts + fold_size if fold < 4 else len(doc_ids)
        test_ids = doc_ids[ts:te]
        train_ids = doc_ids[:ts] + doc_ids[te:]

        tp_stats = compute_tp(train_ids, docs)
        f1, p, r = evaluate_tp(test_ids, docs, tp_stats, theta)
        fold_f1s.append(f1)
        fold_ps.append(p)
        fold_rs.append(r)

    avg_f1 = np.mean(fold_f1s)
    std_f1 = np.std(fold_f1s)
    avg_p = np.mean(fold_ps)
    avg_r = np.mean(fold_rs)
    sweep_results.append({
        'theta': theta, 'f1': avg_f1, 'std': std_f1,
        'precision': avg_p, 'recall': avg_r
    })
    print(f"  θ={theta:.2f}  F1={avg_f1:.4f} (±{std_f1:.4f})  P={avg_p:.4f}  R={avg_r:.4f}")

best = max(sweep_results, key=lambda x: x['f1'])
print(f"\n  Best: θ={best['theta']:.2f}  F1={best['f1']:.4f}")

# Fine-grained sweep around best
print(f"\n  Fine sweep...")
fine_results = []
for theta in np.arange(max(0.05, best['theta']-0.15),
                        min(0.99, best['theta']+0.15), 0.01):
    fold_f1s = []
    for fold in range(5):
        ts = fold * fold_size
        te = ts + fold_size if fold < 4 else len(doc_ids)
        test_ids = doc_ids[ts:te]
        train_ids = doc_ids[:ts] + doc_ids[te:]
        tp_stats = compute_tp(train_ids, docs)
        f1, _, _ = evaluate_tp(test_ids, docs, tp_stats, theta)
        fold_f1s.append(f1)
    fine_results.append({'theta': theta, 'f1': np.mean(fold_f1s)})

best_fine = max(fine_results, key=lambda x: x['f1'])
print(f"  Best (fine): θ={best_fine['theta']:.2f}  F1={best_fine['f1']:.4f}")


Documents for segmentation: 5896
Total signs: 512,910
Total words: 166,379
Avg signs/word: 3.08

  WORD BOUNDARY INFERENCE — HITTITE (5-fold CV)
  θ=0.10  F1=0.5917 (±0.0045)  P=0.4387  R=0.9085
  θ=0.20  F1=0.5540 (±0.0035)  P=0.3860  R=0.9814
  θ=0.30  F1=0.5164 (±0.0021)  P=0.3492  R=0.9906
  θ=0.40  F1=0.5021 (±0.0022)  P=0.3359  R=0.9942
  θ=0.50  F1=0.4901 (±0.0025)  P=0.3251  R=0.9950
  θ=0.60  F1=0.4889 (±0.0024)  P=0.3237  R=0.9988
  θ=0.70  F1=0.4857 (±0.0026)  P=0.3208  R=0.9998
  θ=0.80  F1=0.4846 (±0.0021)  P=0.3198  R=1.0000
  θ=0.90  F1=0.4842 (±0.0021)  P=0.3194  R=1.0000
  θ=0.95  F1=0.4831 (±0.0026)  P=0.3185  R=1.0000

  Best: θ=0.10  F1=0.5917

  Fine sweep...
  Best (fine): θ=0.06  F1=0.6194


## 8. Embedding Comparison (FastText)

Train fastText on Latin (char n-gram 2–5) vs Unicode (char n-gram 1–5).


In [22]:
# ============================================================
# FASTTEXT EMBEDDINGS
# ============================================================
from gensim.models import FastText

for repr_name, doc_dict in [('Latin', docs_latin), ('Unicode', docs_unicode)]:
    sentences = [doc.split() for doc in doc_dict.values() if doc]
    total_tokens = sum(len(s) for s in sentences)

    if repr_name == 'Latin':
        min_n, max_n = 2, 5
    else:
        min_n, max_n = 1, 5

    model = FastText(
        sentences=sentences,
        vector_size=100,
        window=5,
        min_count=2,
        min_n=min_n,
        max_n=max_n,
        epochs=20,
        workers=4
    )

    print(f"{repr_name}: {total_tokens:,} tokens, {len(model.wv):,} vocab, "
          f"char n-gram {min_n}-{max_n}")

    # Save for later use
    if repr_name == 'Latin':
        ft_latin = model
    else:
        ft_unicode = model

print("\nFastText models trained.")


Latin: 178,177 tokens, 11,750 vocab, char n-gram 2-5
Unicode: 516,073 tokens, 336 vocab, char n-gram 1-5

FastText models trained.


## 9. POS Classification

Character n-gram logistic regression: Latin vs Unicode vs Concatenated.


In [23]:
# ============================================================
# POS CLASSIFICATION — HITTITE
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from scipy.sparse import hstack

# Filter to clean Unicode words and sufficient POS classes
hitt_pos = hitt[hitt['clean'] & (hitt['pos_unified'] != 'X')].copy()

# Remove rare POS classes
pos_counts = hitt_pos['pos_unified'].value_counts()
valid_pos = pos_counts[pos_counts >= 20].index
hitt_pos = hitt_pos[hitt_pos['pos_unified'].isin(valid_pos)].copy()

print(f"Tokens for POS classification: {len(hitt_pos)}")
print(f"POS classes: {len(valid_pos)}")
print(f"Distribution:")
for pos, cnt in hitt_pos['pos_unified'].value_counts().items():
    print(f"  {pos:10s} {cnt:6,}")

# Prepare features
X_latin_text = hitt_pos['form_latin'].fillna('').astype(str)
X_unicode_text = hitt_pos['form_unicode'].fillna('').astype(str)
y = hitt_pos['pos_unified'].values

# Vectorizers
vec_latin = TfidfVectorizer(analyzer='char', ngram_range=(2, 5), max_features=50000)
vec_unicode = TfidfVectorizer(analyzer='char', ngram_range=(1, 5), max_features=50000)

X_latin = vec_latin.fit_transform(X_latin_text)
X_unicode = vec_unicode.fit_transform(X_unicode_text)
X_concat = hstack([X_latin, X_unicode])

# 5-fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
for name, X in [('Latin', X_latin), ('Unicode', X_unicode), ('Concat', X_concat)]:
    f1s = []
    for train_idx, test_idx in skf.split(X, y):
        clf = LogisticRegression(max_iter=1000, C=1.0, solver='saga', n_jobs=-1)
        clf.fit(X[train_idx], y[train_idx])
        pred = clf.predict(X[test_idx])
        f1 = f1_score(y[test_idx], pred, average='macro')
        f1s.append(f1)
    results[name] = {'mean': np.mean(f1s), 'std': np.std(f1s), 'folds': f1s}
    print(f"  {name:10s}  F1={np.mean(f1s):.4f} (±{np.std(f1s):.4f})")

print(f"\n  Concat vs best single: Δ = {results['Concat']['mean'] - max(results['Latin']['mean'], results['Unicode']['mean']):+.4f}")


Tokens for POS classification: 126822
POS classes: 6
Distribution:
  NOUN       74,749
  VERB       43,449
  MOD         4,050
  CONJ        2,484
  ADP         1,573
  ADV           517
  Latin       F1=0.8777 (±0.0060)
  Unicode     F1=0.9066 (±0.0017)
  Concat      F1=0.9098 (±0.0020)

  Concat vs best single: Δ = +0.0032


## 10. Lemmatization

Character-level LSTM seq2seq: form → lemma. Evaluate exact match rate.


In [24]:
# ============================================================
# LEMMATIZATION — HITTITE
# ============================================================
import torch
import torch.nn as nn

# Prepare data
hitt_lem = hitt[hitt['clean'] & hitt['lemma'].notna()].copy()
hitt_lem = hitt_lem[hitt_lem['lemma'].astype(str).str.len() > 0].copy()
print(f"Lemmatization tokens: {len(hitt_lem)}")

# Character vocabulary
def build_char_vocab(texts):
    chars = set()
    for t in texts:
        chars.update(str(t))
    vocab = {c: i+2 for i, c in enumerate(sorted(chars))}
    vocab['<PAD>'] = 0
    vocab['<UNK>'] = 1
    return vocab

def encode(text, vocab, max_len=50):
    ids = [vocab.get(c, 1) for c in str(text)[:max_len]]
    ids += [0] * (max_len - len(ids))
    return ids

# Simple seq2seq
class CharSeq2Seq(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.enc = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.dec = nn.LSTM(emb_dim, hidden*2, batch_first=True)
        self.out = nn.Linear(hidden*2, vocab_size)

    def forward(self, src, tgt):
        enc_out, (h, c) = self.enc(self.emb(src))
        h = torch.cat([h[0], h[1]], dim=-1).unsqueeze(0)
        c = torch.cat([c[0], c[1]], dim=-1).unsqueeze(0)
        dec_out, _ = self.dec(self.emb(tgt), (h, c))
        return self.out(dec_out)

# Train and evaluate for both representations
for repr_name, form_col in [('Latin', 'form_latin'), ('Unicode', 'form_unicode')]:
    forms = hitt_lem[form_col].astype(str).tolist()
    lemmas = hitt_lem['lemma'].astype(str).tolist()

    src_vocab = build_char_vocab(forms)
    tgt_vocab = build_char_vocab(lemmas)
    tgt_inv = {v: k for k, v in tgt_vocab.items()}

    # 80/20 split
    n = len(forms)
    idx = list(range(n))
    random.shuffle(idx)
    split = int(n * 0.8)
    train_idx, test_idx = idx[:split], idx[split:]

    # Encode
    X_train = torch.tensor([encode(forms[i], src_vocab) for i in train_idx])
    Y_train = torch.tensor([encode(lemmas[i], tgt_vocab) for i in train_idx])
    X_test = torch.tensor([encode(forms[i], src_vocab) for i in test_idx])

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = CharSeq2Seq(max(len(src_vocab), len(tgt_vocab))+10).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=0)

    # Train
    model.train()
    batch_size = 128
    for epoch in range(10):
        total_loss = 0
        for i in range(0, len(X_train), batch_size):
            src = X_train[i:i+batch_size].to(device)
            tgt = Y_train[i:i+batch_size].to(device)
            out = model(src, tgt[:, :-1])
            loss = criterion(out.reshape(-1, out.size(-1)), tgt[:, 1:].reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    # Evaluate
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in test_idx:
            src = torch.tensor([encode(forms[i], src_vocab)]).to(device)
            tgt = torch.tensor([encode(lemmas[i], tgt_vocab)]).to(device)
            out = model(src, tgt[:, :-1])
            pred_ids = out.argmax(-1)[0].cpu().tolist()
            pred_lemma = ''.join(tgt_inv.get(c, '') for c in pred_ids).replace('<PAD>', '').strip()
            if pred_lemma == lemmas[i]:
                correct += 1

    em = correct / len(test_idx)
    print(f"  {repr_name:10s}  Exact match: {em:.4f} ({correct}/{len(test_idx)})")


Lemmatization tokens: 168101
  Latin       Exact match: 0.0000 (0/33621)
  Unicode     Exact match: 0.0000 (0/33621)


## 11. Results Summary

In [1]:
# ============================================================
# RESULTS SUMMARY
# ============================================================

print("=" * 60)
print("  HITTITE RESULTS SUMMARY")
print("=" * 60)

print(f"\nCorpus: {len(hitt):,} tokens, {hitt['text_id'].nunique():,} texts")
print(f"Unicode conversion: {clean_words/len(hitt)*100:.1f}% clean words, "
      f"{converted_signs/total_signs*100:.1f}% signs")
print(f"Unique Unicode signs: {len(set(s for signs in hitt['unicode_signs'] for s in signs))}")

print(f"\n--- Word Boundary Inference ---")
print(f"Best F1: {best_fine['f1']:.4f} at θ={best_fine['theta']:.2f}")
print(f"Universal θ=0.5: F1={[r for r in sweep_results if r['theta']==0.5][0]['f1']:.4f}")
print(f"(Compare: AKK=.971, SUX=.972, ELX=.989)")

print(f"\n--- POS Classification (n-gram LR) ---")
for name in ['Latin', 'Unicode', 'Concat']:
    print(f"  {name:10s} F1={results[name]['mean']:.4f}")



  HITTITE RESULTS SUMMARY


NameError: name 'hitt' is not defined